# Step 04 — Building Enrichment with ALKIS Landuse & GFK Codes

**Input:** `data/output/01_building_volumes_filtered.gpkg`  
**Output:** `data/output/04_enriched_building_volume_data.gpkg`

**Steps:**
1. Merge GFK building function codes (DE/EN labels) from codelist CSV
2. Spatial join with 5 ALKIS landuse layers → `ALKIS_Landuse_info` (semicolon-separated tags)
3. Spatial join with OSM buildings → `osm_names`, `osm_building_type`
4. Spatial join with OSM landuse → `osm_landuse_class`, `osm_landuse_name`
5. Merge GFK activity class mapping

In [1]:
import sys
sys.path.insert(0, str(__import__('pathlib').Path('..').resolve()))
from config import *

import geopandas as gpd
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')
print('Config loaded')

Config loaded


## 1. Load buildings and merge GFK function codes

In [2]:
gdf = gpd.read_file(VOLUMES_FILTERED_FILE)
print(f'Loaded {len(gdf):,} buildings. CRS: {gdf.crs}')

if BUILDING_FUNCTION_CODELIST.exists():
    df_codes = pd.read_csv(BUILDING_FUNCTION_CODELIST)
    gdf = gdf.merge(df_codes[['function', 'label_de', 'label_en']], on='function', how='left')
    print(f'Merged function labels. Matched: {gdf["label_en"].notna().sum():,}')
else:
    print('WARNING: building_function_codelist.csv not found — label_en/label_de will be empty')
    gdf['label_de'] = np.nan
    gdf['label_en'] = np.nan

Loaded 663,615 buildings. CRS: EPSG:25832
Merged function labels. Matched: 663,615


## 2. Spatial join with ALKIS landuse layers

In [3]:
def add_tag(existing, tag):
    if pd.isna(existing): return tag
    tags = [t.strip() for t in str(existing).split(';') if t.strip()]
    if tag not in tags: tags.append(tag)
    return ';'.join(tags)

gdf['ALKIS_Landuse_info'] = pd.Series(index=gdf.index, dtype='object')

for tag, filepath in LANDUSE_FILES.items():
    if not filepath.exists():
        print(f'WARNING: {filepath.name} not found — skipping {tag} landuse')
        continue
    landuse = gpd.read_file(filepath)[['geometry']].to_crs(gdf.crs)
    j = gpd.sjoin(gdf[['geometry']].reset_index(names='gdf_idx'), landuse, how='inner', predicate='intersects')
    matched_idx = j['gdf_idx'].unique()
    gdf['ALKIS_Landuse_info'] = gdf['ALKIS_Landuse_info'].astype('object')
    mask = gdf.index.isin(matched_idx)
    gdf.loc[mask, 'ALKIS_Landuse_info'] = gdf.loc[mask, 'ALKIS_Landuse_info'].apply(lambda x: add_tag(x, tag))
    print(f'  {tag}: {mask.sum():,} buildings tagged')

print(gdf['ALKIS_Landuse_info'].value_counts(dropna=False).head(10))

  residential: 568,988 buildings tagged
  commercial: 37,476 buildings tagged
  industrial: 22,606 buildings tagged
  public: 14,993 buildings tagged
  sports: 1,476 buildings tagged
ALKIS_Landuse_info
residential               553408
NaN                        35391
commercial                 26188
industrial                 18372
public                     12219
residential;commercial      9747
residential;industrial      3198
residential;public          1992
sports                      1237
commercial;industrial        622
Name: count, dtype: int64


## 3. Spatial join with OSM buildings (names and types)

In [4]:
if ALL_BUILDINGS_OSM_FILE.exists():
    osm_bld = gpd.read_file(ALL_BUILDINGS_OSM_FILE, layer='buildings').to_crs(gdf.crs)
    osm_named = osm_bld[osm_bld['name'].notna() & (osm_bld['name'].str.strip() != '')].copy()

    j = gpd.sjoin(gdf[['geometry']].reset_index(names='gdf_idx'),
                  osm_named[['name', 'building', 'geometry']], how='left', predicate='intersects')

    names = (j.groupby('gdf_idx')['name']
              .apply(lambda s: ';'.join(sorted(set(str(x).strip() for x in s.dropna() if str(x).strip()))))
              .replace('', np.nan).rename('osm_names'))
    types = (j.groupby('gdf_idx')['building']
              .apply(lambda s: ';'.join(sorted(set(str(x).strip() for x in s.dropna() if str(x).strip()))))
              .replace('', np.nan).rename('osm_building_type'))

    gdf['osm_names']        = gdf.index.to_series().map(names)
    gdf['osm_building_type'] = gdf.index.to_series().map(types)
    print(f'OSM building names matched: {gdf["osm_names"].notna().sum():,}')
else:
    gdf['osm_names'] = np.nan
    gdf['osm_building_type'] = np.nan
    print('WARNING: OSM buildings file not found — run notebook 02 first')

OSM building names matched: 16,346


## 4. Spatial join with OSM landuse

In [5]:
landuse_osm_file = INPUT_DIR / 'landuse_osm.gpkg'
if landuse_osm_file.exists():
    landuse = gpd.read_file(landuse_osm_file).to_crs(gdf.crs)
    j = gpd.sjoin(gdf[['geometry']].reset_index(names='gdf_idx'),
                  landuse[['fclass', 'name', 'geometry']], how='left', predicate='intersects')
    class_lu = (j.groupby('gdf_idx')['fclass']
                 .apply(lambda s: ';'.join(sorted(set(x for x in s.dropna()))))
                 .replace('', np.nan))
    name_lu  = (j.groupby('gdf_idx')['name']
                 .apply(lambda s: ';'.join(sorted(set(str(x).strip() for x in s.dropna() if str(x).strip()))))
                 .replace('', np.nan))
    gdf['osm_landuse_class'] = gdf.index.to_series().map(class_lu)
    gdf['osm_landuse_name']  = gdf.index.to_series().map(name_lu)
else:
    gdf['osm_landuse_class'] = np.nan
    gdf['osm_landuse_name']  = np.nan
    print('Optional: place OSM landuse file at data/input/landuse_osm.gpkg for additional enrichment')

## 5. Merge GFK activity class mapping

In [6]:
if ALKIS_ACTIVITY_MAP.exists():
    df_map = pd.read_excel(ALKIS_ACTIVITY_MAP)
    if '_cluster' in gdf.columns:
        gdf = gdf.drop(columns=['_cluster'])
    gdf = gdf.merge(df_map, left_on='function', right_on='gfk_code', how='left').drop(columns=['gfk_code'], errors='ignore')
    print(f'GFK activity map merged. Rows with gfk_class: {gdf["gfk_class"].notna().sum():,}')
else:
    gdf['gfk_class'] = np.nan
    gdf['gfk_name']  = np.nan
    print('WARNING: alkis_building_activity_map.xlsx not found')

GFK activity map merged. Rows with gfk_class: 663,615


## 6. Save

In [7]:
if '_cluster' in gdf.columns:
    gdf = gdf.drop(columns=['_cluster'])

gdf.to_file(ENRICHED_BUILDINGS_FILE, driver='GPKG')
print(f'Saved {len(gdf):,} enriched buildings → {ENRICHED_BUILDINGS_FILE}')

Saved 663,615 enriched buildings → C:\Users\Mayur Patel\Documents\GitHub\Capacity_Calculation-pipeline\capacity-pipeline-generic\data\output\04_enriched_building_volume_data.gpkg
